"""
Hybrid monthly sales forecasting:
1) Multivariate regression for external drivers.
2) ARIMA on regression residuals to capture autocorrelation / temporal structure.
3) Forecast confidence intervals.
4) Explainability layer for driver impacts.

Expected input file:
    synthetic_monthly_sales.csv

Columns:
    date, sales, marketing_spend, avg_price, economic_index, holiday_flag, competitor_promo_index
"""

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

In [7]:
# ---------------------------
# Configuration
# ---------------------------
#BASE_DIR = Path(__file__).resolve().parent
DATA_PATH =  "synthetic_monthly_sales.csv"
TRAIN_RATIO = 0.80
ARIMA_ORDER = (1, 0, 1)  # Example order; tune in a real project via AIC/grid search
FORECAST_HORIZON = 6      # Demonstration horizon from the end of the dataset


In [9]:
# ---------------------------
# Utility functions
# ---------------------------
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add deterministic seasonal / trend features."""
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    out = out.sort_values("date").reset_index(drop=True)
    out["t"] = np.arange(len(out))
    out["month_num"] = out["date"].dt.month

    # Cyclical month encoding
    out["month_sin"] = np.sin(2 * np.pi * out["month_num"] / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * out["month_num"] / 12.0)
    return out


def rmse(y_true: pd.Series, y_pred: pd.Series) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def build_feature_matrix(df: pd.DataFrame) -> list[str]:
    return [
        "marketing_spend",
        "avg_price",
        "economic_index",
        "holiday_flag",
        "competitor_promo_index",
        "t",
        "month_sin",
        "month_cos",
    ]


def fit_regression(train: pd.DataFrame, feature_cols: list[str]):
    """Fit an interpretable OLS regression for coefficient-level explainability."""
    X = sm.add_constant(train[feature_cols], has_constant="add")
    y = train["sales"]
    model = sm.OLS(y, X).fit()
    return model


def fit_arima_on_residuals(train_residuals: pd.Series):
    """Fit ARIMA to regression residuals."""
    model = ARIMA(train_residuals, order=ARIMA_ORDER)
    return model.fit()


def evaluate_forecast(actual: pd.Series, predicted: pd.Series) -> dict:
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": rmse(actual, predicted),
    }


def explain_regression_effects(ols_model, feature_cols: list[str]) -> pd.DataFrame:
    """Return coefficient summary sorted by absolute magnitude."""
    coefs = pd.Series(ols_model.params, index=ols_model.params.index)
    explanation = (
        pd.DataFrame({
            "feature": coefs.index,
            "coefficient": coefs.values,
            "abs_coefficient": coefs.abs().values,
        })
        .query("feature != 'const'")
        .sort_values("abs_coefficient", ascending=False)
        .drop(columns=["abs_coefficient"])
        .reset_index(drop=True)
    )
    return explanation


def standardized_feature_importance(train: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """
    Model-agnostic explainability for the regression component using permutation importance
    on a standardized linear regression pipeline.
    """
    X = train[feature_cols]
    y = train["sales"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ])
    model.fit(X, y)

    result = permutation_importance(
        model,
        X,
        y,
        n_repeats=20,
        random_state=42,
        scoring="neg_mean_absolute_error",
    )

    imp = pd.DataFrame({
        "feature": feature_cols,
        "permutation_importance_mean": result.importances_mean,
        "permutation_importance_std": result.importances_std,
    }).sort_values("permutation_importance_mean", ascending=False)
    return imp


def driver_contributions(ols_model, df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """
    Explain each forecast period by decomposing the regression component into
    per-driver contributions. Intercept is shown separately.
    """
    X = sm.add_constant(df[feature_cols], has_constant="add")
    contrib = pd.DataFrame(index=df.index)
    for col in X.columns:
        contrib[f"contrib_{col}"] = X[col] * ols_model.params[col]
    contrib["regression_prediction"] = contrib.sum(axis=1)
    return contrib

In [11]:
# ---------------------------
# Main pipeline
# ---------------------------
def main():
    # Load and prepare data
    df = pd.read_csv(DATA_PATH)
    df = add_time_features(df)

    feature_cols = build_feature_matrix(df)
    split_idx = int(len(df) * TRAIN_RATIO)
    train = df.iloc[:split_idx].copy()
    test = df.iloc[split_idx:].copy()

    # ---------------------------
    # Step 1: Regression on exogenous drivers
    # ---------------------------
    ols_model = fit_regression(train, feature_cols)

    train_X = sm.add_constant(train[feature_cols], has_constant="add")
    test_X = sm.add_constant(test[feature_cols], has_constant="add")

    train["regression_pred"] = ols_model.predict(train_X)
    test["regression_pred"] = ols_model.predict(test_X)

    train["residual"] = train["sales"] - train["regression_pred"]

    # ---------------------------
    # Step 2: ARIMA on residuals
    # ---------------------------
    arima_model = fit_arima_on_residuals(train["residual"])

    arima_forecast = arima_model.get_forecast(steps=len(test))
    test["residual_forecast"] = arima_forecast.predicted_mean.values

    ci = arima_forecast.conf_int(alpha=0.05)
    # Column names vary slightly depending on statsmodels version
    lower_col = ci.columns[0]
    upper_col = ci.columns[1]

    # ---------------------------
    # Step 3: Hybrid forecast
    # ---------------------------
    test["hybrid_forecast"] = test["regression_pred"] + test["residual_forecast"]
    test["forecast_lower_95"] = test["regression_pred"] + ci[lower_col].values
    test["forecast_upper_95"] = test["regression_pred"] + ci[upper_col].values

    metrics = evaluate_forecast(test["sales"], test["hybrid_forecast"])

    # ---------------------------
    # Step 4: Explainability layer
    # ---------------------------
    coef_summary = explain_regression_effects(ols_model, feature_cols)
    perm_importance = standardized_feature_importance(train, feature_cols)

    test_contrib = driver_contributions(ols_model, test, feature_cols)
    explainable_forecast = pd.concat(
        [
            test[["date", "sales", "regression_pred", "residual_forecast", "hybrid_forecast",
                  "forecast_lower_95", "forecast_upper_95"]].reset_index(drop=True),
            test_contrib.reset_index(drop=True),
        ],
        axis=1,
    )

    # ---------------------------
    # Reporting
    # ---------------------------
    print("\n=== Hybrid Forecast Performance on Holdout Set ===")
    print(json.dumps(metrics, indent=2))

    print("\n=== OLS Regression Summary (truncated view in console) ===")
    print(ols_model.summary())

    print("\n=== Regression Coefficients Ranked by Magnitude ===")
    print(coef_summary.to_string(index=False))

    print("\n=== Permutation Importance (Regression Component) ===")
    print(perm_importance.to_string(index=False))

    print("\n=== Holdout Forecast Sample ===")
    display_cols = [
        "date", "sales", "hybrid_forecast", "forecast_lower_95", "forecast_upper_95"
    ]
    print(test[display_cols].head(10).to_string(index=False))

    # Save outputs
    forecast_output = test[
        [
            "date",
            "sales",
            "regression_pred",
            "residual_forecast",
            "hybrid_forecast",
            "forecast_lower_95",
            "forecast_upper_95",
        ]
    ].copy()

    forecast_output.to_csv("hybrid_forecast_output.csv", index=False)
    coef_summary.to_csv("regression_coefficients_explainability.csv", index=False)
    perm_importance.to_csv("permutation_importance_explainability.csv", index=False)
    explainable_forecast.to_csv("forecast_with_driver_contributions.csv", index=False)

    # ---------------------------
    # Optional: create a future forecast using the last FORECAST_HORIZON rows
    # as stand-ins for known future driver plans.
    #
    # In real life, replace this section with true forward-looking values for
    # marketing spend, price, macro indicators, holiday flags, etc.
    # ---------------------------
    future = df.tail(FORECAST_HORIZON).copy()
    future["date"] = pd.date_range(df["date"].max() + pd.offsets.MonthBegin(1), periods=FORECAST_HORIZON, freq="MS")
    future["t"] = np.arange(len(df), len(df) + FORECAST_HORIZON)
    future["month_num"] = future["date"].dt.month
    future["month_sin"] = np.sin(2 * np.pi * future["month_num"] / 12.0)
    future["month_cos"] = np.cos(2 * np.pi * future["month_num"] / 12.0)

    future_X = sm.add_constant(future[feature_cols], has_constant="add")
    future["regression_pred"] = ols_model.predict(future_X)

    future_arima_fc = arima_model.get_forecast(steps=FORECAST_HORIZON)
    future_ci = future_arima_fc.conf_int(alpha=0.05)
    f_lower_col = future_ci.columns[0]
    f_upper_col = future_ci.columns[1]

    future["residual_forecast"] = future_arima_fc.predicted_mean.values
    future["hybrid_forecast"] = future["regression_pred"] + future["residual_forecast"]
    future["forecast_lower_95"] = future["regression_pred"] + future_ci[f_lower_col].values
    future["forecast_upper_95"] = future["regression_pred"] + future_ci[f_upper_col].values

    future_contrib = driver_contributions(ols_model, future, feature_cols)
    future_explainable = pd.concat(
        [
            future[["date", "hybrid_forecast", "forecast_lower_95", "forecast_upper_95"]].reset_index(drop=True),
            future_contrib.reset_index(drop=True),
        ],
        axis=1,
    )
    future_explainable.to_csv("future_forecast_with_explainability.csv", index=False)

    print("\nSaved files:")
    print(" - hybrid_forecast_output.csv")
    print(" - regression_coefficients_explainability.csv")
    print(" - permutation_importance_explainability.csv")
    print(" - forecast_with_driver_contributions.csv")
    print(" - future_forecast_with_explainability.csv")


if __name__ == "__main__":
    main()


=== Hybrid Forecast Performance on Holdout Set ===
{
  "MAE": 90.24064536756465,
  "RMSE": 120.03233248776334
}

=== OLS Regression Summary (truncated view in console) ===
                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.998
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                     3084.
Date:                Sat, 07 Mar 2026   Prob (F-statistic):           2.68e-73
Time:                        08:27:30   Log-Likelihood:                -413.42
No. Observations:                  67   AIC:                             844.8
Df Residuals:                      58   BIC:                             864.7
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    